# HTMLHeaderTextSplitter

`HTMLHeaderTextSplitter`는 HTML 문서를 **헤더 태그**(`<h1>`, `<h2>`, 등)에 따라 분할합니다.

## 주요 특징

- **HTML 구조 인식**: `<h1>`부터 `<h6>`까지의 헤더 태그 인식
- **메타데이터 보존**: 각 청크에 헤더 정보를 메타데이터로 저장
- **웹 페이지 처리**: URL에서 직접 HTML을 로드하여 분할 가능

## 사용 시나리오

- ✅ **웹 크롤링**: 웹 페이지를 구조화된 청크로 분할
- ✅ **HTML 문서 처리**: 기술 문서, 블로그, 위키 등
- ✅ **RAG 시스템**: 웹 콘텐츠 기반 검색 시스템 구축


# 07. HTMLHeaderTextSplitter

## 학습 목표
- `HTMLHeaderTextSplitter`로 `<h1>`, `<h2>`, `<h3>` 태그 기반으로 HTML을 분할해요
- HTML 헤더 정보가 메타데이터로 저장되는 방식을 이해해요
- `WebBaseLoader`와 결합해서 웹 페이지를 구조적으로 처리해요

## 사전 지식
- 06-MarkdownHeaderTextSplitter 완료
- HTML 기본 태그(`<h1>`, `<p>`, `<div>`) 이해

---

> **MarkdownHeaderTextSplitter와의 차이**: 원리는 동일하지만 처리 대상이 다르에요.
> - **MarkdownHeaderTextSplitter**: `.md` 파일의 `#` 헤더
> - **HTMLHeaderTextSplitter**: `.html` 파일 또는 웹 페이지의 `<h1>~<h6>` 태그


> 🔑 **핵심 개념**: `HTMLHeaderTextSplitter`와 `MarkdownHeaderTextSplitter`는 동일한 원리를 공유합니다. 헤더 태그로 섹션을 구분하고, 헤더 정보를 메타데이터로 보존합니다. 처리 대상만 다릅니다(`.md` vs `.html`/웹 URL).

## 1. HTML 문자열 분할


> 🎯 **강의 포인트**: `WebBaseLoader` + `HTMLHeaderTextSplitter` 조합이 웹 콘텐츠 RAG의 기본 패턴입니다. 웹 페이지를 로딩하고, HTML 헤더 구조에 따라 섹션별로 분할하면 검색 정확도가 크게 향상됩니다.

> ⚠️ **자주 하는 실수**: 헤더 태그가 없거나 `<div>`, `<span>`만 사용하는 HTML 페이지에서는 분할이 제대로 되지 않습니다. 크롤링 전 페이지 소스를 확인해서 `<h1>~<h6>` 태그가 실제로 사용되는지 먼저 확인하세요.

> 💡 **실무 팁**: 기술 문서 사이트(공식 문서, 위키)는 대부분 헤더 구조가 잘 정의되어 있어 `HTMLHeaderTextSplitter`가 효과적입니다. 반면 뉴스 기사나 블로그는 헤더 구조가 불규칙해서 `RecursiveCharacterTextSplitter`가 더 적합합니다.

In [1]:
# ============================================================
# TODO: HTMLHeaderTextSplitter로 HTML 문자열을 헤더 태그 기준으로 분할해보세요
# 힌트: HTMLHeaderTextSplitter(headers_to_split_on=[("h1", "Header 1"), ...]) → split_text(html_string)
# 예상 결과: 12개의 청크가 생성되고 각 청크의 메타데이터에 헤더 계층 정보가 포함됩니다
# ============================================================

from langchain_text_splitters import HTMLHeaderTextSplitter

html_string = """
<!DOCTYPE html>
<html>
<body>
    <h1>LangChain 튜토리얼</h1>
    <p>LangChain으로 LLM 애플리케이션을 구축하는 방법을 배웁니다.</p>
    
    <h2>시작하기</h2>
    <p>LangChain 설치부터 시작합니다.</p>
    
    <h3>설치 방법</h3>
    <p>pip를 사용하여 간단히 설치할 수 있습니다.</p>
    
    <h3>기본 설정</h3>
    <p>API 키를 환경변수에 설정합니다.</p>
    
    <h2>주요 기능</h2>
    <p>LangChain의 핵심 기능을 살펴봅니다.</p>
    
    <h3>체인</h3>
    <p>여러 컴포넌트를 연결하여 작업을 수행합니다.</p>
</body>
</html>
"""

# 1단계: 분할할 HTML 헤더 태그 지정
# - h1~h6 태그를 튜플로 정의: (태그명, 메타데이터 키)
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

# 2단계: HTMLHeaderTextSplitter 생성 및 분할
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
html_header_splits = html_splitter.split_text(html_string)

print(f"분할된 청크 개수: {len(html_header_splits)}")
for i, header in enumerate(html_header_splits):
    print(f"\n{'=' * 60}")
    print(f"[Chunk {i}]")
    print(f"메타데이터: {header.metadata}")
    print(f"내용: {header.page_content[:80]}...")
    print('=' * 60)

분할된 청크 개수: 12

[Chunk 0]
메타데이터: {'Header 1': 'LangChain 튜토리얼'}
내용: LangChain 튜토리얼...

[Chunk 1]
메타데이터: {'Header 1': 'LangChain 튜토리얼'}
내용: LangChain으로 LLM 애플리케이션을 구축하는 방법을 배웁니다....

[Chunk 2]
메타데이터: {'Header 1': 'LangChain 튜토리얼', 'Header 2': '시작하기'}
내용: 시작하기...

[Chunk 3]
메타데이터: {'Header 1': 'LangChain 튜토리얼', 'Header 2': '시작하기'}
내용: LangChain 설치부터 시작합니다....

[Chunk 4]
메타데이터: {'Header 1': 'LangChain 튜토리얼', 'Header 2': '시작하기', 'Header 3': '설치 방법'}
내용: 설치 방법...

[Chunk 5]
메타데이터: {'Header 1': 'LangChain 튜토리얼', 'Header 2': '시작하기', 'Header 3': '설치 방법'}
내용: pip를 사용하여 간단히 설치할 수 있습니다....

[Chunk 6]
메타데이터: {'Header 1': 'LangChain 튜토리얼', 'Header 2': '시작하기', 'Header 3': '기본 설정'}
내용: 기본 설정...

[Chunk 7]
메타데이터: {'Header 1': 'LangChain 튜토리얼', 'Header 2': '시작하기', 'Header 3': '기본 설정'}
내용: API 키를 환경변수에 설정합니다....

[Chunk 8]
메타데이터: {'Header 1': 'LangChain 튜토리얼', 'Header 2': '주요 기능'}
내용: 주요 기능...

[Chunk 9]
메타데이터: {'Header 1': 'LangChain 튜토리얼', 'Header 2': '주요 기능'}
내용: LangChain의 핵심 기능을 살펴봅니다....

[Chunk 10

## 핵심 정리 및 마무리

### HTML vs Markdown 분할 비교

| 항목 | MarkdownHeaderTextSplitter | HTMLHeaderTextSplitter |
|------|:-------------------------:|:----------------------:|
| 처리 형식 | `.md` 파일 | `.html` 파일, 웹 페이지 URL |
| 구분자 | `#`, `##`, `###` | `<h1>`, `<h2>`, `<h3>` |
| 메타데이터 | 동일하게 저장됨 | 동일하게 저장됨 |

### WebBaseLoader와의 결합
> 웹 페이지를 크롤링할 때 `WebBaseLoader`로 로딩한 후 `HTMLHeaderTextSplitter`로 분할하면, 섹션 구조를 유지하면서 처리할 수 있어요. 특히 기술 문서 사이트(docs.python.org 등)에서 효과적이에요.

---

## 다음 예고

Chunking 섹션의 마지막 노트북이에요.

- **08-RecursiveJsonSplitter**: JSON 구조를 보존하면서 분할 + 전체 Chunking 전략 선택 의사결정 트리
